# Deploy router routes

1. Load `.env`
2. Inspect `routes/router_graph.json` (solubility category + all endpoint nodes)
3. `ensure_schema()` + `seed_router_db()` into Postgres / Apache AGE
4. Health-check seeded endpoints

**Prereqs (separate terminal):**

```bash
pixi run serve-settings   # settings on :8011
# Local mat-gram API (max_ports): pixi run api → http://127.0.0.1:8080 then POST /load
# Local Postgres + Apache AGE (see deploy/age/README.md):
#   cd deploy/age && docker compose up -d
# ROUTER_DATABASE_URL must point at 127.0.0.1:5432/string_therapy
```


## 1. Environment

In [5]:
from __future__ import annotations

import json
import os
from pathlib import Path

import httpx
from dotenv import load_dotenv

from string_therapy import ensure_schema, seed_router_db

repo = Path.cwd()
if not (repo / '.env').exists() and (repo.parent / '.env').exists():
    repo = repo.parent

load_dotenv(repo / '.env', override=True)
graph_path = repo / 'routes' / 'router_graph.json'

print('repo:', repo)
print('graph:', graph_path, 'exists=', graph_path.is_file())
print('ROUTER_DATABASE_URL set:', bool(os.getenv('ROUTER_DATABASE_URL')))
print('JWT_AUTH_DISABLED:', os.getenv('JWT_AUTH_DISABLED'))
print('LISETTE_MODEL:', os.getenv('LISETTE_MODEL') or '(unset)')

repo: /root/python_projects/string_therapy_for_material_science
graph: /root/python_projects/string_therapy_for_material_science/routes/router_graph.json exists= True
ROUTER_DATABASE_URL set: True
JWT_AUTH_DISABLED: true
LISETTE_MODEL: gemini/gemini-3.5-flash


## 2. Router graph (solubility category)

In [6]:
graph = json.loads(graph_path.read_text(encoding='utf-8'))
categories = [n for n in graph['nodes'] if n.get('node_type') == 'category']
endpoints = [n for n in graph['nodes'] if n.get('node_type') == 'endpoint']

print('categories:', [(c['id'], c['intent']) for c in categories])
print(f'endpoints ({len(endpoints)}):')
for ep in endpoints:
    print(f"  {ep['id']:2}  {ep['intent']:36}  {ep['method'].upper():4}  {ep['url']}")
print('edges:', len(graph.get('edges', [])))
print('acl rows:', len(graph.get('acl', [])))

categories: [(1, 'solubility'), (3, 'zinc')]
endpoints (3):
   2  solubility_embeddings                 POST  https://mat-gram01-production.up.railway.app/embeddings
   4  zinc_query                            POST  https://queryagent-production-424d.up.railway.app/chat
   5  zinc_capabilities                     POST  https://queryagent-production-424d.up.railway.app/sqlCapabilities
edges: 3
acl rows: 0


## 3. Deploy graph into AGE

`seed_router_db` replaces the `router` graph + ACL from JSON.

In [7]:
ensure_schema()
seed_router_db(graph_path)
print('seeded', graph_path)

seeded /root/python_projects/string_therapy_for_material_science/routes/router_graph.json


## 4. Health-check all viz services

Expect mat-gram API / settings services as needed.


In [8]:
BASES = []
for ep in endpoints:
    # http://127.0.0.1:PORT/...
    parts = ep['url'].split('/')
    base = '/'.join(parts[:3])
    BASES.append((ep['intent'], base, ep['url'], ep['method']))

with httpx.Client(timeout=10.0) as client:
    for intent, base, url, method in BASES:
        try:
            r = client.get(f'{base}/health')
            print(f'{intent:36} GET /health -> {r.status_code} {r.text[:100]}')
        except Exception as exc:
            print(f'{intent:36} GET /health FAILED: {exc}')


solubility_embeddings                GET /health -> 200 {"status":"ok"}
zinc_query                           GET /health -> 200 {"status":"ok","dataset":"zinc_leadlike","twins":6,"lancedb_uri":"/app/data/lancedb","twin_index":"z
zinc_capabilities                    GET /health -> 200 {"status":"ok","dataset":"zinc_leadlike","twins":6,"lancedb_uri":"/app/data/lancedb","twin_index":"z


## 5. POST each route

Confirms every solubility endpoint returns JSON (Plotly payload for the new services).

In [ ]:
payload = {
    'message': 'Deploy check for solubility visualizations',
    'response_format': 'json',
}

with httpx.Client(timeout=30.0) as client:
    for intent, base, url, method in BASES:
        try:
            r = client.request(method.upper(), url, json=payload)
            data = r.json() if r.headers.get('content-type', '').startswith('application/json') else {}
            keys = sorted(data.keys()) if isinstance(data, dict) else type(data).__name__
            has_plotly = isinstance(data, dict) and 'plotly' in data
            has_plot = isinstance(data, dict) and 'plot' in data
            print(
                f'{intent:36} {method.upper():4} -> {r.status_code}'
                f'  plotly={has_plotly} plot={has_plot} keys={keys}'
            )
        except Exception as exc:
            print(f'{intent:36} {method.upper():4} FAILED: {exc}')

## Done

Graph is seeded and every endpoint listed under the **solubility** category was exercised.
Restart `pixi run serve-settings` after editing `services/`.
